# COME 02 — Draft and expert-review masks for AMB82-Mini frames

1. Harmonise each raw frame and draft a mask (RBDI normalised over the sky disk + Otsu threshold).
2. Queue drafts for review, most ambiguous first.
3. Review each draft in an OpenCV window (run Jupyter locally, not on a remote server):
   left mouse = paint cloud, right click = toggle paint/erase, wheel = brush size, **s** = save edits, **a** = accept draft, **Esc** = skip.
4. Recompute cloud fraction and oktas from the validated masks → `amb82_metadata.csv` (used by notebook 01).


In [ ]:
from pathlib import Path
import datetime as dt, math
import cv2, numpy as np, pandas as pd

RAW  = Path("/home/champion/Documents/1.Books/research/Thesis/Weather/WriteUps/PHASE 2/Codes/New Dataset/Circular")
WORK = Path("/home/champion/Documents/1.Books/research/Thesis/Weather/WriteUps/PHASE 2/Codes/amb82_come")
REVIEWER = "GMet_expert"
for d in ("harmonised", "drafts", "masks"): (WORK / d).mkdir(parents=True, exist_ok=True)


In [ ]:
"""Image geometry and the Red-Blue Difference Index (RBDI).

Identical to the preprocessing used for the reported results:
square centre crop -> 480x480 -> circular sky disk (radius 480/2 - 23 px) -> RBDI normalised over the disk.
All functions are numpy/OpenCV only.
"""
import cv2
import numpy as np

HARM = 480          # harmonised side length (pixels)
DISK_MARGIN = 23    # pixels trimmed from the disk radius to exclude the lens rim
MODEL_SIZE = 320    # network input side length


def sky_disk(size=HARM, margin=DISK_MARGIN):
    """Boolean mask of the circular sky disk."""
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    return ((xx - size / 2) ** 2 + (yy - size / 2) ** 2) <= r * r


def harmonise(img, msk=None, size=HARM):
    """Square centre crop and resize to `size`; returns (img, msk, disk)."""
    h, w = img.shape[:2]
    s = min(h, w)
    oy, ox = (h - s) // 2, (w - s) // 2
    img = cv2.resize(img[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_AREA)
    if msk is not None:
        msk = cv2.resize(msk[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_NEAREST)
    return img, msk, sky_disk(size)


def rbdi(img_bgr, disk):
    """Return (float image in [0,1], RBDI = (R-B) min-max normalised over the sky disk)."""
    f = img_bgr.astype(np.float32) / 255.0
    rb = f[..., 2] - f[..., 0]
    lo, hi = rb[disk].min(), rb[disk].max()
    return f, (rb - lo) / max(1e-6, hi - lo)


def model_input(img_bgr, msk=None, model_size=MODEL_SIZE):
    """Build the 4-channel (B, G, R, RBDI) network input exactly as in training.

    Returns x (4, S, S) float32, y (1, S, S) float32 or None, d (1, S, S) bool sky disk.
    """
    img, msk, disk = harmonise(img_bgr, msk)
    f, I = rbdi(img, disk)
    x480 = np.stack([f[..., 0], f[..., 1], f[..., 2], I], -1).astype(np.float32)
    x = cv2.resize(x480, (model_size, model_size), interpolation=cv2.INTER_AREA).transpose(2, 0, 1).copy()
    d = cv2.resize(disk.astype(np.uint8) * 255, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127
    y = None
    if msk is not None:
        y = (cv2.resize(msk, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32)[None]
    return x, y, d[None]


def solid_angle_weights(size=HARM, margin=DISK_MARGIN):
    """Per-pixel solid-angle weights for an equidistant fisheye (zero outside the disk)."""
    disk = sky_disk(size, margin)
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    rr = np.sqrt((xx - size / 2) ** 2 + (yy - size / 2) ** 2)
    k = (np.pi / 2) / r
    w = k * np.sin(k * np.clip(rr, 0, r)) / np.clip(rr, 1e-6, r)
    w[~disk] = 0.0
    return w


In [ ]:
"""Cloud fraction and oktas, computed INSIDE the sky disk only.

Counting the black corners outside the fisheye circle as clear sky would cap the result at
about pi/4 * 8 = 6.3 oktas, so the denominator here is always the number of sky-disk pixels.
"""
import math
import numpy as np


def cloud_fraction(cloud_mask, disk, weights=None):
    cloud = (np.asarray(cloud_mask) > 0) & disk
    if weights is None:
        return float(cloud.sum() / max(1, disk.sum()))
    return float((cloud * weights).sum() / max(1e-12, (disk * weights).sum()))


def to_okta(fraction):
    """floor(8 * fraction), capped at 8 (the convention used throughout the study)."""
    return int(min(8, math.floor(8 * fraction + 1e-9)))


In [ ]:
def draft(I, disk):
    u8 = (np.clip(I, 0, 1) * 255).astype(np.uint8); u8[~disk] = 0
    thr, _ = cv2.threshold(u8[disk].reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = ((u8 > thr) & disk).astype(np.uint8)
    ambiguity = float(((u8[disk] > thr - 12) & (u8[disk] < thr + 12)).mean())   # share of pixels near the threshold
    return m, ambiguity, float(thr)

rows, w = [], solid_angle_weights()
for p in sorted(x for x in RAW.iterdir() if x.suffix.lower() in {".jpg", ".jpeg", ".png"}):
    iid = "amb82_" + p.stem
    img, _, disk = harmonise(cv2.imread(str(p)))
    _, I = rbdi(img, disk)
    m, amb, thr = draft(I, disk)
    cv2.imwrite(str(WORK / "harmonised" / f"{iid}.jpg"), img)
    cv2.imwrite(str(WORK / "drafts" / f"{iid}.png"), m * 255)
    rows.append(dict(id=iid, otsu_threshold=thr, ambiguity=round(amb, 4)))
queue = pd.DataFrame(rows).sort_values("ambiguity", ascending=False)
queue.to_csv(WORK / "review_queue.csv", index=False)
print(f"{len(queue)} drafts; review queue ordered by ambiguity"); queue.head()


In [ ]:
class Editor:
    def __init__(self, img, m): self.img, self.m, self.br, self.draw, self.mode = img, m.copy(), 15, False, 1
    def cb(self, ev, x, y, fl, _):
        if ev == cv2.EVENT_LBUTTONDOWN: self.draw = True; cv2.circle(self.m, (x, y), self.br, self.mode, -1)
        elif ev == cv2.EVENT_MOUSEMOVE and self.draw: cv2.circle(self.m, (x, y), self.br, self.mode, -1)
        elif ev == cv2.EVENT_LBUTTONUP: self.draw = False
        elif ev == cv2.EVENT_RBUTTONDOWN: self.mode = 1 - self.mode
        elif ev == cv2.EVENT_MOUSEWHEEL: self.br = max(3, min(60, self.br + (1 if fl > 0 else -1)))
    def run(self, title):
        cv2.namedWindow(title); cv2.setMouseCallback(title, self.cb)
        while True:
            ov = self.img.copy(); sel = self.m == 1
            ov[sel] = (ov[sel] * 0.6 + np.array([0, 255, 0]) * 0.4).astype(np.uint8)
            cv2.imshow(title, ov); k = cv2.waitKey(20) & 0xFF
            if k == ord("s"): return self.m, True
            if k == ord("a"): return self.m, False
            if k == 27: return None, False

log = []
for r in pd.read_csv(WORK / "review_queue.csv").itertuples():
    if (WORK / "masks" / f"{r.id}.png").exists(): continue          # resume support
    img = cv2.imread(str(WORK / "harmonised" / f"{r.id}.jpg"))
    m0 = cv2.imread(str(WORK / "drafts" / f"{r.id}.png"), 0) // 255
    m1, edited = Editor(img, m0).run(f"{r.id}  s=save  a=accept  esc=skip"); cv2.destroyAllWindows()
    if m1 is None: continue
    cv2.imwrite(str(WORK / "masks" / f"{r.id}.png"), (m1 * 255).astype(np.uint8))
    log.append(dict(id=r.id, edited=edited, reviewer=REVIEWER, ts=dt.datetime.now().isoformat()))
if log:
    f = WORK / "correction_log.csv"; pd.DataFrame(log).to_csv(f, mode="a", index=False, header=not f.exists())
print(f"reviewed this session: {len(log)}")


In [ ]:
disk, w, rows = sky_disk(), solid_angle_weights(), []
for p in sorted((WORK / "masks").glob("*.png")):
    m = cv2.imread(str(p), 0) > 127
    f, fsa = cloud_fraction(m, disk), cloud_fraction(m, disk, w)
    rows.append(dict(id=p.stem, cloud_fraction=round(f, 4), okta=to_okta(f), cloud_fraction_sa=round(fsa, 4), okta_sa=to_okta(fsa)))
meta = pd.DataFrame(rows); meta.to_csv(WORK / "amb82_metadata.csv", index=False)
n = len(pd.read_csv(WORK / "review_queue.csv"))
print(f"validated masks: {len(meta)}/{n}"); print(meta.okta_sa.value_counts().sort_index().to_string())
